In [1]:
from src.dm import DataModule

dm = DataModule(batch_size=4, num_workers=0, pin_memory=False)

dm.setup()

In [2]:
dm.test_df

,image,AOD
0,data/test_images/test_1.tif,1.442470
1,data/test_images/test_2.tif,0.910374
2,data/test_images/test_3.tif,0.500745
3,data/test_images/test_4.tif,0.999882
4,data/test_images/test_5.tif,1.668206
...,...,...
1484,data/test_images/test_1485.tif,1.597977
1485,data/test_images/test_1486.tif,1.819262
1486,data/test_images/test_1487.tif,1.334963
1487,data/test_images/test_1488.tif,0.231196


In [3]:
import os

os.listdir('checkpoints')

['baseline-epoch=4-v1.ckpt',
 'baseline-epoch=29.ckpt',
 'baseline-epoch=4.ckpt',
 'baseline-val_metric=0.76286-epoch=29.ckpt']

In [4]:
import torch 
from src.module import Module

name = "baseline-val_metric=0.76286-epoch=29.ckpt"
checkpoint = f'./checkpoints/{name}'


state_dict = torch.load(checkpoint, map_location='cpu')['state_dict']
module = Module()
module.load_state_dict(state_dict)

/home/juan/miniconda3/envs/peo/lib/python3.8/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/home/juan/miniconda3/envs/peo/lib/python3.8/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedba

<All keys matched successfully>

In [6]:
from tqdm import tqdm

device = "cuda:1"
module.to(device)
module.eval()

preds = []
for batch in tqdm(dm.test_dataloader()):
	dict_of_tensors, _ = batch
	dict_of_tensors = {k: v.to(device) for k, v in dict_of_tensors.items()}
	with torch.no_grad():
		output = module(dict_of_tensors)
		output = output * dm.aod_stats[1] + dm.aod_stats[0] 
		preds += output.cpu().tolist()

100%|██████████| 373/373 [00:25<00:00, 14.85it/s]


,image,AOD
0,data/test_images/test_1.tif,0.103970
1,data/test_images/test_2.tif,0.281763
2,data/test_images/test_3.tif,0.277119
3,data/test_images/test_4.tif,0.141682
4,data/test_images/test_5.tif,0.117333
...,...,...
1484,data/test_images/test_1485.tif,0.248995
1485,data/test_images/test_1486.tif,0.140446
1486,data/test_images/test_1487.tif,0.165951
1487,data/test_images/test_1488.tif,0.223361


In [7]:
dm.test_df['AOD'] = preds
dm.test_df.image = dm.test_df.image.apply(lambda x: x.split('/')[-1])

dm.test_df

,image,AOD
0,test_1.tif,0.103970
1,test_2.tif,0.281763
2,test_3.tif,0.277119
3,test_4.tif,0.141682
4,test_5.tif,0.117333
...,...,...
1484,test_1485.tif,0.248995
1485,test_1486.tif,0.140446
1486,test_1487.tif,0.165951
1487,test_1488.tif,0.223361


In [8]:
dm.test_df.to_csv('submission.csv', index=False, header=False)